In [1]:
# Lab type: extend
# Course: AI401 — AI Applications with LLMs
# Lesson: Observability and Failure Detection in LLM Systems
# Task: Extend a structured logging and metrics system with sliding-window alerting

In [2]:
# Install the Anthropic library
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 11.5 MB/s eta 0:00:00


To use the Anthropic API, you'll need an API key. If you don't already have one, create a key on the [Anthropic console](https://console.anthropic.com/settings/keys).

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `ANTHROPIC_API_KEY`. Then pass the key to the client initialization.

In [3]:
import os

try:
    # Attempt to import google.colab.userdata, which is only available in Colab
    from google.colab import userdata

    # Fetch the API key from Colab's secrets manager
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded from Colab secrets.")
except ImportError:
    # If not in Colab, try to load from a .env file using python-dotenv
    try:
        from dotenv import load_dotenv
        load_dotenv()
        ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
        if ANTHROPIC_API_KEY:
            print("Anthropic API key loaded from .env file.")
        else:
            print("Anthropic API key not found in .env file. Please ensure ANTHROPIC_API_KEY is set.")
    except ImportError:
        print("python-dotenv not installed. Please install it (`pip install python-dotenv`) or ensure ANTHROPIC_API_KEY is set as an environment variable.")
        ANTHROPIC_API_KEY = None

# Ensure the API key is not None before proceeding, or handle the error appropriately
if ANTHROPIC_API_KEY is None:
    raise ValueError("ANTHROPIC_API_KEY is not set. Please set it in Colab secrets or a .env file.")

Anthropic API key loaded from Colab secrets.


# Lab: Extending the LLM Observability Stack

The baseline gives you a structured request logger and a `ValidationWindowTracker` that records validation pass/fail results in a time-based sliding window.

**Your task:** Implement three extensions:

1. `failure_rate(window_size)` on `ValidationWindowTracker` — fraction of recent calls that failed
2. `triage_alert()` — classify an elevated failure rate into a probable root cause
3. `call_with_backoff()` — async retry wrapper with exponential backoff and jitter

## Baseline (working — do not modify)

In [4]:
import asyncio
import hashlib
import json
import random
import time
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
import anthropic


# ---------------------------------------------------------------------------
# Structured request logger
# ---------------------------------------------------------------------------

def log_llm_call(
    *,
    model: str,
    system_prompt: str,
    input_tokens: int,
    output_tokens: int,
    latency_ms: float,
    validation_passed: bool,
    validation_error: str | None,
    request_id: str,
) -> dict:
    """
    Emit a structured log entry for an LLM call.
    Returns the entry dict (for testing).
    In production, print(json.dumps(entry)) ships to your log aggregator.
    """
    prompt_hash = hashlib.sha256(system_prompt.encode()).hexdigest()[:12]
    entry = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "event": "llm_call",
        "request_id": request_id,
        "model": model,
        "prompt_hash": prompt_hash,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_ms": round(latency_ms, 1),
        "validation_passed": validation_passed,
        "validation_error": validation_error,
    }
    # In notebooks: print to see the structured output
    # In production: send to your log aggregator
    print(json.dumps(entry))
    return entry


# ---------------------------------------------------------------------------
# Validation window tracker — baseline (record() only, no failure_rate yet)
# ---------------------------------------------------------------------------

@dataclass
class ValidationWindowTracker:
    """
    Track validation results over a sliding time window.
    Thread-safe for single-process use.
    """
    window_minutes: int = 60
    alert_threshold: float = 0.05
    _entries: deque = field(default_factory=deque)

    def record(self, passed: bool) -> None:
        """Record a validation result, evicting entries outside the window."""
        now = datetime.now(timezone.utc)
        self._entries.append((now, passed))
        cutoff = now - timedelta(minutes=self.window_minutes)
        while self._entries and self._entries[0][0] < cutoff:
            self._entries.popleft()

    # Extension 1 goes here ↓
    # def failure_rate(self, window_size: int = 100) -> float: ...
    # def should_alert(self) -> bool: ...


# Smoke-test baseline
tracker = ValidationWindowTracker()
for passed in [True, True, False, True, False]:
    tracker.record(passed)
print(f'Entries recorded: {len(tracker._entries)}')

Entries recorded: 5


## Extension 1: `failure_rate()` and `should_alert()`

Add two methods to `ValidationWindowTracker`:

- `failure_rate(window_size=100)` — considers only the last `window_size` entries (not the time window), returns float 0.0–1.0
- `should_alert()` — returns `True` if `failure_rate()` exceeds `alert_threshold`

In [6]:
class ValidationWindowTrackerV2(ValidationWindowTracker):
    """
    ValidationWindowTracker with failure rate alerting.
    """

    def failure_rate(self, window_size: int = 100) -> float:
        """
        Return the failure rate over the last window_size entries.
        Returns 0.0 if there are no entries.
        """
        # Get the last window_size entries from the deque
        # _entries stores (timestamp, passed) tuples
        recent_entries = list(self._entries)[-window_size:]

        if not recent_entries:
            return 0.0

        # Count failures (where 'passed' is False)
        failures = sum(1 for _, passed in recent_entries if not passed)
        total_entries = len(recent_entries)

        return failures / total_entries

    def should_alert(self) -> bool:
        """
        Return True if failure_rate() exceeds alert_threshold.
        """
        return self.failure_rate() > self.alert_threshold


# Verify
t = ValidationWindowTrackerV2(alert_threshold=0.05)
# Record 100 entries: 94 pass, 6 fail (6% failure rate)
for i in range(100):
    t.record(i >= 6)   # first 6 fail, rest pass

print(f'Failure rate : {t.failure_rate():.2%} (expect 6.00%)')
print(f'Should alert : {t.should_alert()} (expect True)')

# Sliding window: add 100 more all-passing entries
for _ in range(100):
    t.record(True)

print(f'After 100 passes — failure rate: {t.failure_rate():.2%} (expect 0.00%)')
print(f'Should alert : {t.should_alert()} (expect False)')
print('Extension 1: PASS')

Failure rate : 6.00% (expect 6.00%)
Should alert : True (expect True)
After 100 passes — failure rate: 0.00% (expect 0.00%)
Should alert : False (expect False)
Extension 1: PASS


## Extension 2: `triage_alert()`

Implement the root-cause triage function. It should return one of:
- `"PROVIDER_ISSUE"` — HTTP error rate > 10%
- `"PROMPT_REGRESSION"` — prompt hash changed recently AND schema failure rate > 3%
- `"DATA_DRIFT"` — semantic failure rate > 5% AND schema failure rate < 2%
- `"UNKNOWN"` — none of the above match

In [8]:
def triage_alert(
    http_error_rate: float,
    schema_failure_rate: float,
    semantic_failure_rate: float,
    prompt_hash_changed: bool,
) -> str:
    """
    Classify an elevated failure rate into a probable root cause.
    Returns a triage string for the on-call alert.
    """
    if http_error_rate > 0.10:
        return 'PROVIDER_ISSUE'
    elif prompt_hash_changed and schema_failure_rate > 0.03:
        return 'PROMPT_REGRESSION'
    elif semantic_failure_rate > 0.05 and schema_failure_rate < 0.02:
        return 'DATA_DRIFT'
    else:
        return 'UNKNOWN'


# Verify
cases = [
    # (http_err, schema_fail, semantic_fail, hash_changed, expected)
    (0.15, 0.01, 0.01, False, 'PROVIDER_ISSUE'),
    (0.01, 0.08, 0.02, True,  'PROMPT_REGRESSION'),
    (0.01, 0.01, 0.08, False, 'DATA_DRIFT'),
    (0.01, 0.01, 0.01, False, 'UNKNOWN'),
]
all_pass = True
for http_err, schema_fail, semantic_fail, hash_changed, expected in cases:
    result = triage_alert(http_err, schema_fail, semantic_fail, hash_changed)
    ok = expected in result
    all_pass = all_pass and ok
    print(f'{"PASS" if ok else "FAIL"}: got {result!r}, expected {expected!r}')
if all_pass:
    print('Extension 2: PASS')

PASS: got 'PROVIDER_ISSUE', expected 'PROVIDER_ISSUE'
PASS: got 'PROMPT_REGRESSION', expected 'PROMPT_REGRESSION'
PASS: got 'DATA_DRIFT', expected 'DATA_DRIFT'
PASS: got 'UNKNOWN', expected 'UNKNOWN'
Extension 2: PASS


## Extension 3: `call_with_backoff()`

Implement an async wrapper that retries on `anthropic.RateLimitError` (429) and 5xx `anthropic.APIStatusError` using exponential backoff with jitter.

- Base wait: `2 ** attempt` seconds
- Jitter: add `random.uniform(0, 1)` to the wait
- Max retries: configurable (default 4)
- Re-raise on the final attempt

In [12]:
import asyncio
import random
import anthropic
import httpx # Import httpx for a more accurate mock response

async def call_with_backoff(
    client: anthropic.AsyncAnthropic,
    *,
    model: str,
    system: str,
    messages: list,
    max_tokens: int,
    max_retries: int = 4,
) -> anthropic.types.Message:
    """
    Call the Anthropic API with exponential backoff on 429 and 5xx.
    """
    for attempt in range(max_retries + 1):
        try:
            response = await client.messages.create(
                model=model,
                system=system,
                messages=messages,
                max_tokens=max_tokens,
            )
            return response
        except (anthropic.RateLimitError, anthropic.APIStatusError) as e:
            # Determine if this error is retryable
            is_retryable = False
            if isinstance(e, anthropic.RateLimitError):
                is_retryable = True # RateLimitError (429) is always retryable
            elif isinstance(e, anthropic.APIStatusError) and (500 <= e.status_code < 600):
                is_retryable = True # 5xx APIStatusError is retryable

            if not is_retryable:
                # If it's not a retryable error, re-raise it immediately
                raise e

            # If it is retryable, proceed with backoff logic
            if attempt == max_retries:
                print(f"Max retries reached. Re-raising exception: {e}")
                raise e

            wait_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"Attempt {attempt + 1} failed. Retrying in {wait_time:.2f} seconds...")
            await asyncio.sleep(wait_time)

    # This part should theoretically not be reached if max_retries is handled correctly
    raise RuntimeError("Failed after all retries without re-raising an explicit exception.")


# Verify: simulate RateLimitError on first two attempts
async def test_backoff():
    call_log = []

    class FakeMessage:
        content = [type('C', (), {'text': 'ok'})()]

    class FakeClient:
        class messages:
            @staticmethod
            async def create(**kwargs):
                call_log.append(len(call_log))
                if len(call_log) <= 2:
                    # Create a mock httpx.Request object
                    mock_request = httpx.Request("GET", "http://mock-api.com")
                    # Create a mock httpx.Response object including the request
                    mock_response = httpx.Response(status_code=429, request=mock_request, headers={})
                    raise anthropic.RateLimitError(
                        message='rate limited',
                        response=mock_response,
                        body={},
                    )
                return FakeMessage()

    result = await call_with_backoff(
        FakeClient(),
        model='claude-haiku-4-5-20251001',
        system='test',
        messages=[],
        max_tokens=64,
        max_retries=2 # Set max_retries to 2 to match the test case of 3 attempts (0, 1, 2)
    )
    print(f'Succeeded after {len(call_log)} attempts (expect 3)')
    assert len(call_log) == 3, f'Expected 3 attempts, got {len(call_log)}'
    print('Extension 3: PASS')

# To run an async function in Colab, use await directly if an event loop is already running.
# The RuntimeError 'asyncio.run() cannot be called from a running event loop' is common in notebooks.
await test_backoff()

Attempt 1 failed. Retrying in 1.16 seconds...
Attempt 2 failed. Retrying in 2.40 seconds...
Succeeded after 3 attempts (expect 3)
Extension 3: PASS
